# 01 — Mean-variance: minimum variance (GMV) and maximum Sharpe ratio (MSR)

GMV and MSR as Python classes (`GMV` and `MaxSharpe` in `maplab.models`; the text writes MSR, code and figures `MaxSharpe`), scored by the rolling backtest against the two benchmarks used in every notebook: equal weight (EW) and 60/40 (SPY/IEF).

## §1 Why mean-variance is fragile

GMV is risk-based: its weights depend on Σ alone. MSR is return-based: it also needs expected returns μ, here the trailing one-year mean. Both invert Σ, so they amplify estimation error. The optimizer loads on assets whose μ is overstated, shorts or drops their close substitutes, and can rotate the whole portfolio on a small change in μ; hence the "error maximizer". A year of daily data pins down Σ reasonably well, but not μ: the standard error of a one-year mean is the asset's own volatility (§4.3). Every model here therefore has to beat two portfolios that estimate nothing: EW and 60/40. Notebook 03 tries a better μ, 04 a better Σ, and 05–07 drop μ.

## §2 The optimization problems

Unconstrained, both portfolios have closed forms:

$$w_{\text{GMV}} = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}'\Sigma^{-1}\mathbf{1}}, \qquad w_{\text{MSR}} = \frac{\Sigma^{-1}(\mu - r_f\mathbf{1})}{\mathbf{1}'\Sigma^{-1}(\mu - r_f\mathbf{1})}.$$

On this universe the unconstrained MSR takes large offsetting long and short positions (§4.2), so the classes solve the long-only versions:

$$\text{GMV:}\quad \min_{w}\; w'\Sigma w \quad \text{s.t.}\quad \mathbf{1}'w = 1,\; w \ge 0,$$

$$\text{MSR:}\quad \max_{w}\; \frac{(\mu - r_f\mathbf{1})'w}{\sqrt{w'\Sigma w}} \quad \text{s.t.}\quad \mathbf{1}'w = 1,\; w \ge 0.$$

**Inputs**, from the 252 trading days before each rebalance: $\Sigma$ is the annualized sample covariance of log returns; $\mu = \overline{r}_{\log} \times 252 + \tfrac12\,\mathrm{diag}(\Sigma)$ converts the log mean into an arithmetic mean; $r_f$ is the trailing mean of BIL.

## Setup

The 13 ETFs, BIL as the risk-free rate and the conventions of notebook 00; log returns feed μ and Σ, simple returns compound the backtest.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
from maplab import Panel, apply_style, GMV, MaxSharpe, EqualWeight, FixedWeight, backtest, summary
from maplab.models import gmv_closed_form, tangency_closed_form, _tangency_long_only

import _nb01_helpers as h

In [ ]:
apply_style()
prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf = ml.load_rf_returns()                      # BIL, risk-free only
rf_daily = rf["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": rf})

## §3 The classes and the rolling backtest

Calling a model at a rebalance date runs three steps: estimate μ and Σ from the 252 trading days before the date (the §2 inputs; `Strategy._estimate` in `src/maplab/models.py`), solve the §2 problem, and validate the weights (no missing values, sum to 1, no negative weights for long-only classes). Every model in this repo shares the first and last steps.

**Solvers.** Both long-only problems are solved numerically. MSR uses the convex form $\min_y y'\Sigma y$ s.t. $(\mu - r_f\mathbf{1})'y = 1$, $y \ge 0$, then sets $w = y/\mathbf{1}'y$. If no asset has $\mu_i > r_f$, there is no positive Sharpe ratio to maximize and the class falls back to GMV.

**Backtest** (`ml.backtest`), identical for every model: rebalance at each month-end using only earlier data (2022-12-31 uses data through 12-30 and trades on 2023-01-03); pay 10 bp per unit of one-way turnover; let weights drift between rebalances; score from 2009-01-31, 208 rebalances in all.

## §4 Snapshot at the train/test split

One rebalance date, 2022-12-31: what the classes see and what they do with it.

### §4.1 Inputs at the snapshot

The estimation window (the 252 trading days 2021-12-31 to 2022-12-30), $r_f$, μ and volatility per asset, and UUP's correlations with the other 12.

In [ ]:
rdates = ml.rebalance_dates(simple_returns.index)
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
split_rebal = rdates[rdates <= split_ts].max()

mu, Sigma = GMV()._estimate(panel, split_rebal)
rf_ann = MaxSharpe()._rf_ann(panel, split_rebal)

print(f"rf (trailing 252-day BIL mean, annualized): {rf_ann:.2%}\n")

vol = pd.Series(np.sqrt(np.diag(Sigma.to_numpy())), index=Sigma.index)
inputs = pd.DataFrame({"μ (%)": mu, "vol (%)": vol, "μ − r_f (%)": mu - rf_ann}).rename_axis(None)
inputs = inputs.sort_values("μ − r_f (%)", ascending=False)
display(ml.reporting.show_table(inputs, pct=list(inputs.columns), pct_dp=2))

corr = Sigma / np.outer(vol, vol)
print(f"corr(EFA, EEM): {corr.loc['EFA', 'EEM']:.2f}")
uup_corr = corr["UUP"].drop("UUP").sort_values().rename_axis(None).to_frame("corr with UUP")
ml.reporting.show_table(uup_corr, dp=2)

**Reading.** The window is the 2022 rate shock: only DBC (+19.3%) and UUP (+7.6%) have μ above $r_f$, GLD sits just below (−0.3%) and the other ten are negative. UUP also has the lowest volatility (9.27%, just under TIP's 9.34%) and is negatively correlated with every other asset (−0.22 to −0.75). MSR allocates as if 2022's returns continue.

### §4.2 Long-only vs unconstrained weights

The classes' long-only weights next to the closed forms of §2 on the same inputs. The tangency closed form exists only if the unconstrained GMV's expected return exceeds $r_f$; here it does (2.04% against 1.39%).

In [ ]:
w_gmv = GMV()(panel, split_rebal)
w_msr = MaxSharpe()(panel, split_rebal)
w_gmv_unc = gmv_closed_form(Sigma)

mu_gmv_unc = float(w_gmv_unc @ mu)
assert mu_gmv_unc > rf_ann, "tangency closed form is undefined: unconstrained GMV return is not above r_f"
w_tan = tangency_closed_form(mu, Sigma, rf=rf_ann)

weights = pd.DataFrame({
    "GMV long-only": w_gmv,
    "MSR long-only": w_msr,
    "GMV unconstrained": w_gmv_unc,
    "MSR unconstrained": w_tan,
}).rename_axis(None)
print("weights (%):")
ml.reporting.show_table(weights.sort_values("MSR unconstrained", ascending=False), pct=list(weights.columns), pct_dp=1)

The four portfolios as bars (grey: unconstrained closed form; colour: the class's long-only weights; own y-axis per panel), with gross leverage, total short weight and largest absolute weight below.

In [ ]:
fig, leverage_table = h.unconstrained_vs_long_only(Sigma, mu, rf_ann, w_gmv, w_msr)
plt.show()

lt = leverage_table
shown = lt.astype(object)
shown.loc["gross leverage Σ|w|"] = lt.loc["gross leverage Σ|w|"].map(lambda v: f"{v:.2f}×")
for row in ("sum of negative weights", "max |w|"):
    shown.loc[row] = (lt.loc[row] * 100).map(lambda v: f"{v:.1f}%")
print(shown.to_string())

**Reading.** Long-only, GMV holds UUP 58%, EFA 17%, IEF 14% and GLD 11%, and MSR holds UUP 74%, DBC 23% and GLD 3%. Unconstrained, MSR runs 13.0× gross: $\Sigma^{-1}$ turns small μ gaps between correlated assets into offsetting bets, so EFA and EEM (correlation 0.82, 7.6 points apart in μ) end at +165% and −126%, while GMV, with no μ to bet on, stays at 1.6×. That is estimation error at full strength, and the reason the classes are long-only.

### §4.3 Sensitivity to μ

Perturb μ with draws from $N(0, s^2\Sigma)$ and re-solve MSR, 200 draws per noise level $s$. That is the sampling error of a one-year mean: the standard errors equal the annualized volatilities, and the errors are correlated like the returns. GMV does not use μ and is unchanged by construction.

In [ ]:
n_draws = 200
w_msr_base = w_msr.to_numpy()
Sigma_np = Sigma.to_numpy()

rows = []
for scale in (0.25, 0.5, 1.0):
    rng = np.random.default_rng(20260924)
    for _ in range(n_draws):
        eps = rng.multivariate_normal(np.zeros(len(mu)), Sigma_np)  # one-year sampling error of mu: N(0, Sigma)
        excess = mu.to_numpy() + scale * eps - rf_ann
        if excess.max() <= 0:
            continue
        w_m, _ = _tangency_long_only(excess, Sigma_np, "MSR", split_rebal)
        rows.append({
            "noise": scale,
            "half_L1": 0.5 * np.abs(w_m - w_msr_base).sum(),
            "max_w": w_m.max(),
            "holdings": int((w_m > 1e-4).sum()),
            "top": mu.index[int(np.argmax(w_m))],
        })
sens = pd.DataFrame(rows)

In [ ]:
table = sens.groupby("noise").agg(**{
    "draws": ("half_L1", "size"),
    "median ½L1 (%)": ("half_L1", "median"),
    "p90 ½L1 (%)": ("half_L1", lambda s: s.quantile(0.9)),
    "median max weight (%)": ("max_w", "median"),
    "median holdings": ("holdings", "median"),
    "UUP on top (%)": ("top", lambda s: (s == "UUP").mean()),
})
table.index = table.index.map(lambda s: f"{s:.2f}")
table.index.name = "s"

print("MSR weight change vs the unperturbed solution (noise in units of the one-year std error of mu):")
ml.reporting.show_table(table, pct=["median ½L1 (%)", "p90 ½L1 (%)", "median max weight (%)", "UUP on top (%)"],
                        pct_dp=1, int_cols=["draws", "median holdings"])

**Reading.** At one standard error the median draw reallocates 25% of MSR's portfolio (53% at the 90th percentile) and UUP loses the top spot in 15% of draws; even at a quarter of that noise the median draw moves 8%. The median draw holds three assets at every level (199 draws at $s = 1$: one left no asset above $r_f$). This is the error maximizer of §1; with μ re-estimated monthly it shows up as turnover in §5.

### §4.4 The long-only efficient frontier

The long-only frontier on the §4.1 inputs, the capital market line through MSR, and the benchmarks. The grey cloud is 20,000 random long-only portfolios drawn from a Dirichlet(0.3); α < 1 concentrates the draws, so the cloud reaches the corners where GMV and MSR sit.

In [ ]:
mc_alpha = 0.3
fig = h.frontier_figure(Sigma, mu, w_gmv, w_msr, mc_alpha, rf_ann, split_rebal)
plt.show()

**Reading.** On 2022's inputs EW and 60/40 sit inside the cloud with negative expected returns, while MSR sits where the capital market line touches the frontier. That ranking is only as good as one year of μ; §5 shows what happened next.

### §4.5 Capital vs risk

Capital weights (dark) against each asset's share of portfolio variance (light) at 2022-12-31; a negative share means the asset hedges the rest. The table counts diversification three ways: effective N $= 1/\sum_i w_i^2$, the number of equally weighted assets with the same concentration; uncorrelated bets, the exponential of the entropy of the portfolio's variance shares across Σ's principal portfolios (between 1 and 13); and the diversification ratio $w'\sigma/\sigma_p$. For long-only GMV the two bars coincide for every held asset: at the optimum every held asset has the same marginal risk, so its variance share equals its weight.

In [ ]:
weights_snap = {
    "GMV": w_gmv,
    "MaxSharpe": w_msr,
    "EW": EqualWeight()(panel, split_rebal),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, split_rebal),
}
div = ml.diagnostics.diversification_table(weights_snap, Sigma).rename(columns={
    "w_UUP": "UUP weight (%)", "effN": "effective N", "n_uncorr": "uncorrelated bets",
    "max_RC_share": "largest risk share (%)", "DR": "diversification ratio", "exante_vol": "ex-ante vol (%)"})
display(ml.reporting.show_table(div, pct=["UUP weight (%)", "largest risk share (%)", "ex-ante vol (%)"], dp=2))

fig = ml.plotting.capital_vs_risk(weights_snap, Sigma, title=f"Capital weight vs risk share, {split_rebal.date()} (sample Σ)")
plt.show()

**Reading.** Holding many assets is not holding many bets: EW's 13 assets amount to about 1.4 uncorrelated bets, and SPY carries 89% of 60/40's variance for 60% of its capital. MSR concentrates both ways (UUP is 74% of capital and 57% of variance), while GMV has the highest diversification ratio (3.69) and about 7.6 uncorrelated bets from four assets. The bets are counted over principal portfolios, not assets, so they can exceed the number of holdings.

## §5 Out-of-sample results

Walk-forward, as set out in notebook 00: at each month-end the class re-estimates μ and Σ on the prior 252 days and holds the weights for a month, so every return is out of sample; "train" and "test" split one track record at 2022-12-31. The cell checks the Sharpe ratios against `registrations/reproduction.toml`, the values later notebooks build on.

In [ ]:
strategies = {
    "GMV": GMV(),
    "MaxSharpe": MaxSharpe(),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag}

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    windows = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in windows.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

full = summary_table.xs("full", level="window")
train = summary_table.xs("train (<= split)", level="window")
test = summary_table.xs("test (> split)", level="window")
overview = pd.DataFrame({
    "Sharpe full": full["sharpe"], "Sharpe train": train["sharpe"], "Sharpe test": test["sharpe"],
    "return (%)": full["ann_return"], "vol (%)": full["ann_vol"],
    "max DD (%)": full["max_dd"], "turnover (%/yr)": full["ann_turnover"],
}).rename(index={"EqualWeight": "EW"}).rename_axis(None)
ml.reporting.show_table(overview, pct=["return (%)", "vol (%)", "max DD (%)", "turnover (%/yr)"],
                        pct_dp=1, dp=2, int_cols=["turnover (%/yr)"])

In [ ]:
ml.inference.check_reproduction(summary_table, {
    "GMV": "GMV_sample", "MaxSharpe": "MaxSharpe_sample",
    "EqualWeight": "EW", "60/40": "SixtyForty_SPY_IEF",
})
msr = strategies["MaxSharpe"]
print("reproduction gate passed (GMV, MaxSharpe, EW, 60/40 vs registrations/reproduction.toml, 2 dp); "
      f"MaxSharpe fallbacks {len(msr.fallback_dates)}, retries {len(msr.retry_dates)}")

**Reading.** Over the full sample 60/40 has the highest Sharpe ratio (0.91), ahead of EW (0.74), MSR (0.72) and GMV (0.62); MSR leads only in the 40-month test window (1.13). GMV has the lowest volatility and 60/40 the highest (3.2% vs 10.1%). MSR's 248% annual turnover costs about 0.25% a year at 10 bp, a fraction of its 1.2-point return gap to EW, so costs do not explain its lag; whether any gap is significant is tested in notebook 08.

## §6 What drove the results: allocation through time

### §6.1 Weights by asset group

Target weights summed into the six groups at every rebalance (dotted line: train/test split). EW and 60/40 are fixed and appear only in the table.

In [ ]:
gmv_group = ml.diagnostics.weights_by_group(results["GMV"]["wlog"])
msr_group = ml.diagnostics.weights_by_group(results["MaxSharpe"]["wlog"])

fig = ml.plotting.group_stackplot(results, ["GMV", "MaxSharpe"], figsize=(9, 8), split_ts=split_ts)
plt.show()

print("Mean target weight by asset group (%):")
group_means = pd.DataFrame({
    "GMV": gmv_group.mean(),
    "MaxSharpe": msr_group.mean(),
    "EW": ml.diagnostics.weights_by_group(results["EqualWeight"]["wlog"]).mean(),
    "60/40": ml.diagnostics.weights_by_group(results["60/40"]["wlog"]).mean(),
}).rename_axis(None).rename(index=ml.plotting.GROUP_LABELS)
display(ml.reporting.show_table(group_means, pct=list(group_means.columns), pct_dp=1))

uup_w = results["GMV"]["wlog"]["UUP"]
print(f"\nGMV UUP weight: mean {uup_w.mean():.1%}, min {uup_w.min():.1%}, max {uup_w.max():.1%}")

msr_peaks = pd.DataFrame({"max weight (%)": msr_group.max() * 100,
                          "first date at max": msr_group.idxmax().dt.date}).rename_axis(None).rename(index=ml.plotting.GROUP_LABELS)
print("\nMaxSharpe: peak weight by group")
print(msr_peaks.to_string(float_format=lambda v: f"{v:.1f}"))
near_all = msr_group[msr_group.max(axis=1) >= 0.95]
print(f"\nMaxSharpe rebalances with one group >= 95%: {len(near_all)} of {len(msr_group)}")
for d, row in near_all.iterrows():
    print(f"  {d.date()}  {ml.plotting.GROUP_LABELS.get(row.idxmax(), row.idxmax())}  {row.max():.1%}")

**Reading.** GMV is a dollar-and-bonds portfolio: 37% UUP on average (18–58%), 21% rates, 16% TIPS and only 7% equities, against 23% for EW and 60% for 60/40, so its low return goes with its low volatility. MSR's averages look diversified but hide the path: 95–100% equities in February–April 2014 and August 2017, and 85% in the dollar in October 2022. With μ a trailing one-year mean, MSR rotates into whichever assets had the best year, with nothing to smooth the switch.

### §6.2 Concentration and turnover

Effective N, largest weight and trailing-12-rebalance turnover at every rebalance; EW dashed in the effective-N and turnover panels.

In [ ]:
names = ["GMV", "MaxSharpe", "60/40"]
colors = {name: ml.plotting.run_color(name) for name in names}

fig = ml.plotting.concentration_panel(
    results, names,
    rows=("effN", "max_w", "turnover"), ref="EqualWeight", ref_label="EW", split_ts=split_ts,
    figsize=(9.5, 10.5), colors=colors,
    suptitle="GMV, MaxSharpe and 60/40: effective N, largest weight, turnover",
)
plt.show()

conc = pd.DataFrame({
    name: {
        "median effN": results[name]["wlog"].apply(ml.diagnostics.effective_n, axis=1).median(),
        "median max weight (%)": results[name]["wlog"].max(axis=1).median(),
        "largest max weight (%)": results[name]["wlog"].max(axis=1).max(),
        "ann. turnover (%/yr)": summary_table.loc[(name, "full"), "ann_turnover"],
    }
    for name in ["GMV", "MaxSharpe", "EqualWeight", "60/40"]
}).T.rename(index={"EqualWeight": "EW"})
ml.reporting.show_table(conc, pct=["median max weight (%)", "largest max weight (%)", "ann. turnover (%/yr)"],
                        dp=2, int_cols=["ann. turnover (%/yr)"])

**Reading.** Both optimizers are concentrated (median effective N 3.8 for GMV, 3.0 for MSR, 13 for EW), yet 60/40 is more concentrated still (1.9) and has the best Sharpe, so concentration alone is not the problem. Stability is: GMV's largest weight peaks at 58%, while MSR's reaches 100% and turns over 248% a year. Both converge on UUP into the 2022 split.

## §7 TAKEAWAY — the benchmarks win on Sharpe; the optimizers differ in risk and stability

- **The fixed-weight benchmarks are hard to beat.** Full-sample Sharpe: 60/40 0.91, EW 0.74, MSR 0.72, GMV 0.62; MSR leads only in the 40-month test window (1.13).
- **Risk ranks clearly; cost barely matters.** GMV has the lowest volatility and 60/40 the highest (3.2% vs 10.1%); MSR's 248% annual turnover costs about 0.25% a year.
- **MSR is an estimation-error maximizer.** One standard error of noise in μ moves a median 25% of its weights (§4.3); unconstrained it runs 13× gross, so long-only is not cosmetic.
- **GMV leans on the dollar.** UUP averages 37% of GMV (18–58%), with 7% in equities; it enters as a hedge, negatively correlated with all 12 other assets at the snapshot (§4.1).
- **Many assets are not many bets.** EW's 13 assets are about 1.4 uncorrelated bets, yet 60/40, the most concentrated portfolio, has the best Sharpe; MSR's weakness is instability, not concentration (§6.2).

Sample-mean μ is the weak link: notebook 03 tries a better μ, notebook 04 a better Σ, and notebooks 05–07 drop μ altogether.